# Backward Elimination and Principal Component Analysis

Two ways to fight too many features. Backward elimination drops the statistically insignificant ones; PCA replaces many correlated columns with a few uncorrelated components.

## Imports and loading helper

In [ ]:
# Standard imports for the whole course
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Find the Datasets folder whether the notebook is run from the
# project root or from inside the notebooks folder.
from pathlib import Path
for candidate in [Path("Datasets"), Path("../Datasets")]:
    if candidate.exists():
        DATA = candidate
        break

def load(name):
    """Load a CSV from the Datasets folder by file name."""
    return pd.read_csv(DATA / name)

print("Datasets are in:", DATA.resolve())
print(sorted(p.name for p in DATA.glob("*.csv")))

## Backward elimination on the fifty companies data

A p-value above the significance level means the feature is not earning its place.

In [ ]:
df = load("Multiple_Linear_Regression_50_CompList.csv")
X = df.iloc[:, :-1].values
y = df.iloc[:, 4].values

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

le = LabelEncoder()
X[:, 3] = le.fit_transform(X[:, 3])
ct = ColumnTransformer([("State", OneHotEncoder(), [3])],
                       remainder="passthrough")
X = ct.fit_transform(X)
X = X[:, 1:]

## The OLS summary table

We prepend a column of ones to carry the intercept, then read the p-values from the table.

In [ ]:
try:
    import statsmodels.api as sm
    X_new = np.append(np.ones((50, 1)).astype(int), X, axis=1)
    x_opt = X_new[:, [0, 1, 2, 3, 4, 5]].astype("float")
    model = sm.OLS(endog=y, exog=x_opt).fit()
    print(model.summary())
except ImportError:
    print("statsmodels is not installed; run pip install statsmodels")

## The single surviving predictor

Dropping the weak features leaves research and development spend, and a one feature model is just as accurate.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

x_be = df.iloc[:, 0].values.reshape(-1, 1)
x_train, x_test, y_train, y_test = train_test_split(
    x_be, y, test_size=0.2, random_state=0)
reg = LinearRegression().fit(x_train, y_train)
print("Train score:", reg.score(x_train, y_train))
print("Test score :", reg.score(x_test, y_test))

## PCA on the wine data

With thirteen chemical features a logistic regression is accurate, but two components may do almost as well.

In [ ]:
wine = load("PCA_Wine.csv")
Xw = wine.iloc[:, 0:13].values
yw = wine.iloc[:, 13].values

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
X_train, X_test, y_train, y_test = train_test_split(
    Xw, yw, test_size=0.2, random_state=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

pca = PCA().fit(X_train)
plt.bar(range(1, 14), pca.explained_variance_ratio_ * 100)
plt.plot(range(1, 14), np.cumsum(pca.explained_variance_ratio_) * 100,
         "o-", color="red")
plt.xlabel("Principal component"); plt.ylabel("Variance explained (%)")
plt.show()

## Reducing to two components

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

pca2 = PCA(n_components=2)
X_train2 = pca2.fit_transform(X_train)
X_test2 = pca2.transform(X_test)

clf = LogisticRegression(random_state=0).fit(X_train2, y_train)
print("Accuracy with 2 components:", accuracy_score(y_test, clf.predict(X_test2)))

Reduce the dimension gradually, checking accuracy at each step, rather than jumping straight to a very small number.